リンク：https://docs.databricks.com/aws/ja/generative-ai/agent-framework/stateful-agents

# Databricks Lakebaseを使用した短期記憶を持つエージェントの作成とデプロイ


このノートブックでは、Agent FrameworkとLakebaseをエージェントの永続的なメモリおよびチェックポイントストアとして使用し、短期記憶を持つエージェントを構築する方法を示します。

スレッドを使用すると、会話の状態をLakebaseに保存できるため、完全な会話履歴を送信する代わりにスレッドIDをエージェントに渡すことができます。

このノートブックでは、以下を行います：
1. Databricks Agentでスレッドidを使用して状態を管理するLakebaseを使ったエージェントグラフを作成
2. LangGraphエージェントを`ResponsesAgent`インターフェースでラップし、Databricks機能との互換性を確保
3. エージェントの動作をローカルでテスト
4. Unity Catalogにモデルを登録し、Review App、Playgroundなどで使用するためにエージェントをログおよびデプロイ

## 前提条件
- Lakebaseインスタンスが準備され実行されていること。ドキュメントを参照してください（[AWS](https://docs.databricks.com/aws/en/oltp/create/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/oltp/create/)）。
  - SQL Warehouses -> Lakebase Postgres -> Create database instanceに移動してLakebaseインスタンスを作成できます。このノートブックに記入するには、Lakebaseの「Connection details」セクションから値を取得する必要があります。
- このノートブック全体のすべての「TODO」を完了してください

### 依存関係のインストール

In [0]:
%pip install -U -qqqq databricks-langchain[memory] uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

## 初回セットアップのみ：Lakebaseインスタンスでチェックポイントテーブルをセットアップ

In [0]:
# 初回チェックポイントテーブルのセットアップ
from databricks.sdk import WorkspaceClient
from databricks_langchain import CheckpointSaver

# --- TODO: Lakebaseインスタンス名を入力してください ---
INSTANCE_NAME = "lakebase-name"

# テーブルが存在しない場合は作成
with CheckpointSaver(instance_name=INSTANCE_NAME) as saver:
    saver.setup()           # チェックポイントテーブルをセットアップ
    print("✅ チェックポイントテーブルの準備が完了しました。")

# コードでエージェントを定義

## エージェントコードをファイルagent.pyに書き込む
以下の単一セルでエージェントコードを定義します。これにより、`%%writefile`マジックコマンドを使用してエージェントコードをローカルPythonファイルに書き込み、後続のログ記録とデプロイに使用できます。

## ResponsesAgentインターフェースを使用してLangGraphエージェントをラップ
Databricks AI機能との互換性を確保するため、`LangGraphResponsesAgent`クラスは`ResponsesAgent`インターフェースを実装してLangGraphエージェントをラップします。

Databricksでは、オープンソース標準を使用してマルチターン会話エージェントの作成を簡素化する`ResponsesAgent`の使用を推奨しています。MLflowの[ResponsesAgentドキュメント](https://www.mlflow.org/docs/latest/llms/responses-agent-intro/)を参照してください。

In [0]:
%%writefile agent.py
import logging
import os
import uuid
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    CheckpointSaver,
)
from databricks.sdk import WorkspaceClient
from langchain_core.messages import AIMessage, AIMessageChunk, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
)

logger = logging.getLogger(__name__)
logging.basicConfig(level=os.getenv("LOG_LEVEL", "INFO"))

############################################
# LLMエンドポイントとシステムプロンプトを定義
############################################
# TODO: モデルサービングエンドポイントを置き換えてください
LLM_ENDPOINT_NAME = "databricks-claude-3-7-sonnet"

# TODO: システムプロンプトを更新してください
SYSTEM_PROMPT = "あなたは役に立つアシスタントです。利用可能なツールを使用して質問に答えてください。"

############################################
# Lakebase設定
############################################
# TODO: Lakebaseインスタンス名を入力してください
LAKEBASE_INSTANCE_NAME = "lakebase-name"

###############################################################################
## エージェントのツールを定義し、テキスト生成以外のデータ取得やアクションを
## 実行できるようにします
## さらにツールを作成し、使用例を確認するには、
## https://docs.databricks.com/en/generative-ai/agent-framework/agent-tool.htmlを参照してください
###############################################################################
tools = []

# UCツールの例。必要に応じて追加してください
UC_TOOL_NAMES: list[str] = []
if UC_TOOL_NAMES:
    uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
    tools.extend(uc_toolkit.tools)

# Databricksベクトル検索インデックスをツールとして使用
# https://docs.databricks.com/en/generative-ai/agent-framework/unstructured-retrieval-tools.html#locally-develop-vector-search-retriever-tools-with-ai-bridgeを参照してください
# 非構造化検索用のベクトル検索ツールインスタンスを格納するリスト
VECTOR_SEARCH_TOOLS = []

# ベクトル検索リトリーバーツールを追加するには、
# VectorSearchRetrieverToolとcreate_tool_infoを使用し、
# 結果をTOOL_INFOSに追加します。
# 例：
# VECTOR_SEARCH_TOOLS.append(
#     VectorSearchRetrieverTool(
#         index_name="",
#         # filters="..."
#     )
# )

tools.extend(VECTOR_SEARCH_TOOLS)

#####################
## エージェントロジックを定義
#####################

class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]

class LangGraphResponsesAgent(ResponsesAgent):
    """プールされたLakebaseチェックポイントを使用するステートフルエージェント。"""

    def __init__(self, lakebase_config: dict[str, Any]):
        self.workspace_client = WorkspaceClient()

        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

    def _create_graph(self, checkpointer: Any):
        def should_continue(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1]
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "continue"
            return "end"

        preprocessor = (
            RunnableLambda(lambda state: [{"role": "system", "content": self.system_prompt}] + state["messages"])
            if self.system_prompt
            else RunnableLambda(lambda state: state["messages"])
        )
        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            return {"messages": [response]}

        workflow = StateGraph(AgentState)
        workflow.add_node("agent", RunnableLambda(call_model))

        if tools:
            workflow.add_node("tools", ToolNode(tools))
            workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
            workflow.add_edge("tools", "agent")
        else:
            workflow.add_edge("agent", END)

        workflow.set_entry_point("agent")
        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        """リクエストからthread_idを取得するか、新しいものを作成します。

        優先順位：
        1. custom_inputsに存在する場合はthread_idを使用
        2. 利用可能な場合はチャットコンテキストからconversation_idを使用
        3. 新しいUUIDを生成

        Returns:
            thread_id: この会話に使用するスレッド識別子
        """
        ci = dict(request.custom_inputs or {})

        if "thread_id" in ci:
            return ci["thread_id"]

        # チャットコンテキストからのconversation idをthread idとして使用
        # https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id

        # 新しいthread_idを生成
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        # 受信したResponsesメッセージをChatCompletions形式に変換
        # LangChainはChatCompletionsからLangChain形式に自動的に変換します
        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        langchain_msgs = cc_msgs
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        with CheckpointSaver(instance_name=LAKEBASE_INSTANCE_NAME) as checkpointer:
            graph = self._create_graph(checkpointer)

            for event in graph.stream(
                {"messages": langchain_msgs},
                checkpoint_config,
                stream_mode=["updates", "messages"],
            ):
                if event[0] == "updates":
                    for node_data in event[1].values():
                        if len(node_data.get("messages", [])) > 0:
                            yield from output_to_responses_items_stream(node_data["messages"])
                elif event[0] == "messages":
                    try:
                        chunk = event[1][0]
                        if isinstance(chunk, AIMessageChunk) and chunk.content:
                            yield ResponsesAgentStreamEvent(
                                **self.create_text_delta(delta=chunk.content, item_id=chunk.id),
                            )
                    except Exception as exc:
                        logger.error("チャンクのストリーミングエラー: %s", exc)


# ----- モデルをエクスポート -----
mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(LAKEBASE_INSTANCE_NAME)
mlflow.models.set_model(AGENT)

# エージェントをローカルでテスト

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT
# メッセージ1、thread_idを含めない（新しいスレッドを作成）
result = AGENT.predict({
    "input": [{"role": "user", "content": "I am working on stateful agents"}]
})
print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]

In [0]:
# メッセージ2、スレッドIDを含め、エージェントが前の予測メッセージからのコンテキストを記憶していることを確認
response2 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
})
print("レスポンス2:", response2.model_dump(exclude_none=True))

In [0]:
# thread idを渡さずにエージェントを呼び出す例 - メモリを保持しないことに注意
response3 = AGENT.predict({
    "input": [{"role": "user", "content": "What am I working on?"}],
})
print("レスポンス3 thread idなし:", response3.model_dump(exclude_none=True))

In [0]:
# predict streamの例
for chunk in AGENT.predict_stream({
    "input": [{"role": "user", "content": "What am I working on?"}],
    "custom_inputs": {"thread_id": thread_id}
}):
    print("チャンク:", chunk.model_dump(exclude_none=True))

In [0]:
# ChatContextからのconversation_idをthread_idとして使用する例
# https://mlflow.org/docs/latest/api_reference/python_api/mlflow.types.html#mlflow.types.agent.ChatContext
from agent import AGENT
import mlflow
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ChatContext
)

conversation_id = "e396d36f-b237-484f-ad6e-f000551703f5"

req = ResponsesAgentRequest(
    input=[{"role": "user", "content": "I am working on stateful agents"}],
    context=ChatContext(
        conversation_id=conversation_id,
        user_id="email@databricks.com"
    )
)
result = AGENT.predict(req)

print(result.model_dump(exclude_none=True))
thread_id = result.custom_outputs["thread_id"]
print(f"エージェントから解決されたthread_id: {thread_id}")

# エージェントをMLflowモデルとしてログ記録
agent.pyファイルからコードとしてエージェントをログ記録します。[MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code)を参照してください。

## Databricksリソースの自動認証を有効化
最も一般的なDatabricksリソースタイプについて、Databricksはログ記録時にエージェントのリソース依存関係を事前に宣言することをサポートおよび推奨しています。これにより、エージェントをデプロイする際に自動認証パススルーが有効になります。自動認証パススルーを使用すると、Databricksはエージェントエンドポイント内からこれらのリソース依存関係に安全にアクセスするための短命認証情報を自動的にプロビジョニング、ローテーション、管理します。

自動認証を有効にするには、`mlflow.pyfunc.log_model()`を呼び出す際に依存するDatabricksリソースを指定します。

**TODO:** 
- lakebaseをリソースタイプとして追加
- Unity Catalogツールが[ベクトル検索インデックス](https://docs.databricks.com/docs%20link)をクエリするか、[外部関数](https://docs.databricks.com/docs%20link)を利用する場合、依存するベクトル検索インデックスとUC接続オブジェクトをそれぞれリソースとして含める必要があります。ドキュメントを参照してください（[AWS](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/log-agent#resources)）。

In [0]:
# デプロイ時に自動認証パススルー用に指定するDatabricksリソースを決定
import mlflow
from agent import tools, LLM_ENDPOINT_NAME, LAKEBASE_INSTANCE_NAME
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint, DatabricksLakebase
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(LLM_ENDPOINT_NAME), DatabricksLakebase(database_instance_name=LAKEBASE_INSTANCE_NAME)]

for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "What is an LLM agent?"
        }
    ],
    "custom_inputs": {"thread_id": "example-thread-123"},
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-langchain[memory]=={get_distribution('databricks-langchain[memory]').version}",
        ]
    )

# Agent Evaluationでエージェントを評価
Mosaic AI Agent Evaluationを使用して、期待される応答やその他の評価基準に基づいてエージェントの応答を評価します。指定した評価基準を使用して反復をガイドし、MLflowを使用して計算された品質メトリクスを追跡します。Databricksドキュメントを参照してください（[AWS](https://docs.databricks.com/(https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)）。

ツール呼び出しを評価するには、カスタムメトリクスを追加します。Databricksドキュメントを参照してください（[AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)）。

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # 該当する場合はここにさらにスコアラーを追加
)

# MLfLow UIで評価結果を確認（コンソール出力を参照）

# デプロイ前のエージェント検証
エージェントを登録およびデプロイする前に、mlflow.models.predict() APIを使用してデプロイ前チェックを実行します。

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "I am working on stateful agents"}]},
    env_manager="uv",
)

# Unity Catalogにモデルを登録
MLflowモデルをUnity Catalogに登録するために、以下の`catalog`、`schema`、`model_name`を更新してください。

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: UCモデルのカタログ、スキーマ、モデル名を定義してください
catalog = "catalog"
schema = "schema"
model_name = "short-term-memory-agent"

UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# モデルをUCに登録
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

エージェントをデプロイ

In [0]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "docs"}, deploy_feedback_model=False)

# 次のステップ
エージェントのデプロイが完了するまでに約15分かかります。エージェントがデプロイされた後、Review App/playgroundでチャットして追加のチェックを実行したり、組織内のSMEと共有してフィードバックを得たり、本番アプリケーションに組み込んだりできます。

Lakebaseインスタンスをクエリして、さまざまなスレッド/チェックポイントでの会話の記録を確認できます。最近10件のチェックポイントを確認する基本的なクエリは次のとおりです：

```
SELECT
    c.*,
    (c.checkpoint::json->>'ts')::timestamptz AS ts
FROM checkpoints c
ORDER BY ts DESC
LIMIT 10;
```